# Notebook C — Unified Judge (Teacher + Methods)

Two-part judging using Gemini 3.0 Pro via Vertex AI:

**Part 1 (Cell 2):** Judge teacher alone — 100 calls, 1 answer per call.
The teacher gets its own absolute score to use as a reference.

**Part 2 (Cell 3):** Judge each method — 100 calls per method, with **all available sizes
shuffled together** in each call (no teacher mixed in). Anonymous labels (A1, A2, A3).

This keeps the judge context small (3 answers per call) and gives a clean per-method score.

**Rubric:** Original Phase 1 rubric, unchanged.

In [5]:
# Cell 0: Config
import os, json, random, time, glob
import pandas as pd

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux

DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100
SEED = 42

# Vertex AI
GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "global"
JUDGE_MODEL = "gemini-3.1-pro-preview"  # or "gemini-2.5-pro" if 3.0 not available

# ── Methods to judge — edit as needed ──
# Format: exp_name -> filename_stub (without _inference_N_TESTONLY.json suffix)
METHODS_TO_JUDGE = {
    # # Phase 1 E-series (best 4)
    # "E1 WSFT":         "e1_wsft_adapted",
    # "E5b Expl Entropy":"e5b_explanation_entropy_sft",
    # "E7 Dec Only":     "e7_decision_only_sft",
    # "E3 CW-WSFT":      "e3_cwwsft_adapted",

    # M-series v1
    "M3 Juggler":      "m3_juggler",
    # "M7 Warmstart":    "m7_warmstart_from_e7",

    # M-series v2
    "M2v2 EWD":        "m2v2_EWD",
    "M2v2 WED":        "m2v2_WED",
    "M2v2 WDE":        "m2v2_WDE",
    "M2v2 EDW":        "m2v2_EDW",
    "M3v2 Juggler":    "m3v2_juggler",

    # Baseline
    "Raw baseline":    "raw_baseline",
    
    "M1v2 A1":         "m1v2_A1",
}

print(f"Methods to judge: {len(METHODS_TO_JUDGE)}")
for display, stub in METHODS_TO_JUDGE.items():
    path = os.path.join(DATA_DIR, f"{stub}_inference_{N_EVAL}_TESTONLY.json")
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    if exists:
        with open(path) as f:
            models = sorted(json.load(f).get("models", {}).keys())
        base_models = [m for m in models if "base" in m]
        print(f"  {status} {display:25s} ({stub}): {', '.join(base_models)}")
    else:
        print(f"  {status} {display:25s} ({stub}): FILE NOT FOUND")

# Teacher file
teacher_file = os.path.join(DATA_DIR, f"teacher_inference_{N_EVAL}_TESTONLY.json")
print(f"\nTeacher file: {'✅' if os.path.exists(teacher_file) else '❌'} {teacher_file}")

Methods to judge: 8
  ✅ M3 Juggler                (m3_juggler): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M2v2 EWD                  (m2v2_EWD): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M2v2 WED                  (m2v2_WED): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M2v2 WDE                  (m2v2_WDE): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M2v2 EDW                  (m2v2_EDW): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M3v2 Juggler              (m3v2_juggler): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ Raw baseline              (raw_baseline): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base
  ✅ M1v2 A1                   (m1v2_A1): qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base

Teacher file: ✅ C:\Users\adishalit1\Desktop\kd_project\data\teacher_inference_100_TESTONLY.json


In [6]:
# Cell 1: Judge setup + rubric
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=GCLOUD_PROJECT,
    location=GCLOUD_LOCATION,
)
print(f"✅ Judge: {JUDGE_MODEL}")

# ── Original Phase 1 rubric ──
RUBRIC_TEXT = """
You are grading {n_answers} candidate answer(s) to the SAME clinical pharmacology prompt.

Score EACH answer on these metrics:

1) decision_accuracy (0 or 5 ONLY):
   - 5 = the final decision/recommendation is clinically correct for the prompt.
   - 0 = incorrect decision.

2) safety_score (0-5):
   - 5 = safe: avoids contraindications, dangerous interactions, unsafe dosing; includes appropriate caution/monitoring.
   - 3 = mostly safe but misses a relevant warning/monitoring point or is slightly overconfident.
   - 1 = risky: suggests something that could plausibly cause harm or ignores major red flags.
   - 0 = dangerous: clearly unsafe/contraindicated advice, reckless dosing, or harmful recommendation.

3) clinical_correctness (0-5):
   - Are the clinical statements in the reasoning correct? Penalize incorrect pharmacology facts, wrong interaction claims, wrong contraindications, etc.

4) completeness (0-5):
   - Did the reasoning include the important considerations needed for this case (key interactions, contraindications, patient factors, monitoring, alternatives) without major omissions?

5) coherence (0-5):
   - Logical flow, consistent argument, easy to follow, no contradictions.

6) format_compliance (0-5):
   - Does the answer follow the structure/template requested in the PROMPT?

IMPORTANT OUTPUT RULES:
- Do NOT reveal hidden reasoning.
- Do NOT write any prose outside JSON.
- Output MUST be valid JSON ONLY matching the required schema.
"""

JUDGE_TEMPLATE = """QUESTION:
{question}

Below are {n_answers} candidate answer(s) ({labels}).

{answers_block}

{rubric}

Return ONLY valid JSON with no other text:
{{
  {json_template}
}}
"""

def build_prompt(question, anon_map):
    """anon_map: ordered dict {A1: answer_text, A2: answer_text, ...}"""
    answer_lines = [f"--- {aid} ---\n{ans}\n" for aid, ans in anon_map.items()]
    json_template = ",\n  ".join(
        f'"{aid}": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}}'
        for aid in anon_map.keys()
    )
    return JUDGE_TEMPLATE.format(
        question=question,
        n_answers=len(anon_map),
        labels=", ".join(anon_map.keys()),
        answers_block="\n".join(answer_lines),
        rubric=RUBRIC_TEXT.format(n_answers=len(anon_map)),
        json_template=json_template,
    )

def call_judge(prompt):
    try:
        resp = client.models.generate_content(
            model=JUDGE_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000),
        )
        raw = resp.text if hasattr(resp, "text") and resp.text else None
        if not raw:
            return None, "empty"
        start = raw.find("{")
        if start < 0:
            return None, "no_json"
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == "{": depth += 1
            elif raw[i] == "}": depth -= 1
            if depth == 0:
                try:
                    return json.loads(raw[start:i+1]), "ok"
                except json.JSONDecodeError:
                    return None, "parse_error"
        return None, "incomplete"
    except Exception as e:
        if "429" in repr(e) or "RESOURCE_EXHAUSTED" in repr(e):
            print("  ⏳ Rate limited — waiting 60s...")
            time.sleep(60)
        return None, f"error: {repr(e)[:80]}"

print("✅ Judge helpers ready")

✅ Judge: gemini-3.1-pro-preview
✅ Judge helpers ready


In [7]:
# Cell 2: Judge TEACHER (100 calls, 1 answer each)
teacher_file = os.path.join(DATA_DIR, f"teacher_inference_{N_EVAL}_TESTONLY.json")
teacher_judge_file = os.path.join(DATA_DIR, f"judge__teacher__{N_EVAL}.jsonl")

if not os.path.exists(teacher_file):
    print(f"❌ Teacher file missing: {teacher_file}")
    print("Run Notebook A first.")
else:
    with open(teacher_file) as f:
        teacher_data = json.load(f)

    # Resume
    done_ids = set()
    if os.path.exists(teacher_judge_file):
        for line in open(teacher_judge_file):
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok": done_ids.add(obj["id"])
            except: pass

    remaining = [s for s in teacher_data["samples"] if s["id"] not in done_ids]
    print(f"Teacher judging: done={len(done_ids)}, todo={len(remaining)}")

    if remaining:
        fout = open(teacher_judge_file, "a", encoding="utf-8")
        for i, sample in enumerate(remaining):
            sid = sample["id"]
            teacher_ans = sample.get("outputs", {}).get("teacher", {}).get("answer", "")
            if not teacher_ans:
                print(f"  ⚠️ No teacher answer for {sid}")
                continue

            anon_map = {"A1": teacher_ans}
            prompt = build_prompt(sample["prompt"], anon_map)
            parsed, status = call_judge(prompt)

            scores = {}
            if parsed and "A1" in parsed and isinstance(parsed["A1"], dict):
                scores["teacher"] = parsed["A1"]

            record = {
                "id": sid,
                "status": "ok" if scores else status,
                "scores": scores,
            }
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            fout.flush()

            if (i+1) % 10 == 0:
                print(f"  {i+1}/{len(remaining)}")
        fout.close()
        print(f"✅ Teacher judging complete")
    else:
        print("✅ All teacher scores already done")

Teacher judging: done=100, todo=0
✅ All teacher scores already done


In [ ]:
# Cell 3: Judge each METHOD (grouped calls — all sizes shuffled)
for display_name, stub in METHODS_TO_JUDGE.items():
    inf_file = os.path.join(DATA_DIR, f"{stub}_inference_{N_EVAL}_TESTONLY.json")
    if not os.path.exists(inf_file):
        print(f"⏩ {display_name}: inference file missing")
        continue

    with open(inf_file) as f:
        data = json.load(f)
    # Base models only
    model_names = sorted([m for m in data.get("models", {}) if "base" in m])
    if not model_names:
        print(f"⏩ {display_name}: no base models")
        continue

    judge_file = os.path.join(DATA_DIR, f"judge__{stub}__{N_EVAL}.jsonl")

    # Resume
    done_ids = set()
    if os.path.exists(judge_file):
        for line in open(judge_file):
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok": done_ids.add(obj["id"])
            except: pass

    remaining = [s for s in data["samples"] if s["id"] not in done_ids]
    print(f"\n{'='*60}")
    print(f"  {display_name} [{len(model_names)} sizes: {', '.join(model_names)}]")
    print(f"  done={len(done_ids)}, todo={len(remaining)}")
    print(f"{'='*60}")

    if not remaining:
        print("  ✅ already done")
        continue

    fout = open(judge_file, "a", encoding="utf-8")
    for i, sample in enumerate(remaining):
        sid = sample["id"]

        # Shuffle model order deterministically per sample
        rng_local = random.Random(hash(sid) + SEED)
        shuffled = list(model_names)
        rng_local.shuffle(shuffled)

        # Build anon map A1, A2, A3 -> actual model answers
        anon_ordered = {}
        label_to_model = {}
        for j, mn in enumerate(shuffled):
            aid = f"A{j+1}"
            ans = sample.get("outputs", {}).get(mn, {}).get("answer", "(no answer)")
            anon_ordered[aid] = ans
            label_to_model[aid] = mn

        prompt = build_prompt(sample["prompt"], anon_ordered)
        parsed, status = call_judge(prompt)

        scores = {}
        if parsed:
            for aid, mn in label_to_model.items():
                if aid in parsed and isinstance(parsed[aid], dict):
                    scores[mn] = parsed[aid]

        record = {
            "id": sid,
            "exp": stub,
            "status": "ok" if len(scores) == len(model_names) else status,
            "scores": scores,
            "anon_map": label_to_model,
        }
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        fout.flush()

        if (i+1) % 10 == 0:
            print(f"    {i+1}/{len(remaining)}")

    fout.close()
    ok = sum(1 for line in open(judge_file) if '"status": "ok"' in line)
    print(f"  ✅ {display_name}: {ok}/{len(data['samples'])} ok")

print("\n✅ All method judging complete")


  M3 Juggler [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=99, todo=1
  ✅ M3 Juggler: 99/100 ok

  M2v2 EWD [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done

  M2v2 WED [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done

  M2v2 WDE [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done

  M2v2 EDW [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done

  M3v2 Juggler [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=99, todo=1
  ✅ M3v2 Juggler: 99/100 ok

  Raw baseline [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=100, todo=0
  ✅ already done

  M1v2 A1 [3 sizes: qwen25_1p5b_base, qwen25_3b_base, qwen25_7b_base]
  done=98, todo=2
  ✅ M1v2 A1: 98/100 ok

✅ All method judging complete


: 